In [1]:
import numpy as np

In [ ]:
class DecisionTree:

    # Inisialisasi model dengan kedalaman pohon maksimal 5
    def __init__(self, max_depth=5):
        self.tree = None
        self.max_depth = max_depth

    # Mengambil nilai kelas yang paling sering muncul
    def most_common_value(self, arr):
        unique, counts = np.unique(arr, return_counts=True)
        return unique[np.argmax(counts)]

    # Menghitung Gini Impurity untuk mengukur ketidakmurnian kelas
    def gini_impurity(self, y):
        unique, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs**2)

    # Menghitung Gini Gain untuk mengetahui kualitas pembagian data
    def gini_gain(self, parent, left, right):
        parent_gini = self.gini_impurity(parent)
        left_gini = self.gini_impurity(left)
        right_gini = self.gini_impurity(right)

        # Menghitung bobot masing-masing cabang berdasarkan jumlah data
        left_weight = len(left) / len(parent)
        right_weight = len(right) / len(parent)

        # Menghitung selisih Gini sebelum dan sesudah pembagian
        return parent_gini - (left_weight * left_gini + right_weight * right_gini)

    # Membangun pohon keputusan secara rekursif
    def build_tree(self, x, y, current_depth):

        # Menghentikan percabangan jika mencapai max_depth atau semua kelas sudah sama
        if current_depth >= self.max_depth or len(np.unique(y)) == 1:
            return self.most_common_value(y)

        # Menyiapkan variabel untuk menyimpan split terbaik
        best_feature = None
        best_split = None
        best_gain = -1

        # Melakukan iterasi pada setiap fitur
        for feature in range(x.shape[1]):

            # Mengambil seluruh nilai unik dari fitur
            unique_values = np.unique(x[:, feature])

            # Mencoba setiap nilai sebagai batas pemisah data
            for values in unique_values:

                # Membagi data menjadi cabang kiri dan kanan
                left_mask = x[:, feature] <= values
                right_mask = x[:, feature] > values

                # Melewati pembagian jika salah satu cabang kosong
                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue

                # Menghitung Gini Gain dari pembagian tersebut
                gain = self.gini_gain(y, y[left_mask], y[right_mask])

                # Menyimpan fitur dan nilai split dengan Gini Gain tertinggi
                if gain > best_gain:
                    best_feature = feature
                    best_split = values
                    best_gain = gain

        # Menghentikan percabangan jika tidak ditemukan split yang valid
        if best_feature is None:
            return self.most_common_value(y)

        # Membagi data berdasarkan fitur dan nilai split terbaik
        left_mask = x[:, best_feature] <= best_split
        right_mask = x[:, best_feature] > best_split

        # Membentuk node dan melanjutkan percabangan ke kiri dan kanan
        return {
            'feature': best_feature,
            'split': best_split,
            'left': self.build_tree(x[left_mask], y[left_mask], current_depth+1),
            'right': self.build_tree(x[right_mask], y[right_mask], current_depth+1)
        }

    # Melatih model dan membangun pohon keputusan dari data training
    def fit(self, x, y):
        self.tree = self.build_tree(x, y, 0)

    # Melakukan prediksi pada satu sampel data
    def predict_sample(self, tree, sample):

        # Jika node bukan dictionary, berarti sudah mencapai leaf node
        if not isinstance(tree, dict):
            return tree

        # Mengambil indeks fitur dan nilai batas pemisah dari node
        feature_index = tree['feature']
        split_value = tree['split']

        # Menentukan cabang berdasarkan nilai fitur pada sampel
        if sample[feature_index] <= split_value:
            return self.predict_sample(tree['left'], sample)
        else:
            return self.predict_sample(tree['right'], sample)

    # Melakukan prediksi pada seluruh sampel data
    def predict(self, x):

        predictions = []

        # Melakukan prediksi satu per satu pada setiap sampel
        for sample in x:
            pred = self.predict_sample(self.tree, sample)
            predictions.append(pred)

        # Mengubah hasil prediksi menjadi array NumPy
        return np.array(predictions)

In [ ]:
def stratified_split(df, kolom_label):
    daftar_train = []
    daftar_test = []
    labels = df[kolom_label].unique()
    for label in labels:
        label_train = df[df[kolom_label] == label].sample(random_state=42, frac=0.8, replace=False)
        label_test = df[df[kolom_label] == label].drop(label_train.index)

        daftar_train.append(label_train) 
        daftar_test.append(label_test)

    daftar_train = pd.concat(daftar_train)
    daftar_test = pd.concat(daftar_test)

    x_te = daftar_test.iloc[:,:-1]
    y_te = daftar_test[kolom_label]

    x_tr = daftar_train.iloc[:,:-1]
    y_tr = daftar_train[kolom_label]

    return x_tr.values, y_tr.values, x_te.values, y_te.values

In [ ]:
def grid_search(depths, x_tr, y_tr, x_te, y_te):
    akurasi_train = []
    akurasi_test = []

    for d in depths:
        model = DecisionTree(d)
        model.fit(x_tr, y_tr)
        y_tr_pred = model.predict(x_tr)
        y_te_pred = model.predict(x_te)

        akurasi_train.append(np.mean(y_tr == y_tr_pred))
        akurasi_test.append(np.mean(y_te == y_te_pred))

        print(f'Depth: {d}✔')

    akurasi_df = pd.DataFrame(data = {'Train': akurasi_train, 'Test': akurasi_test}, index=depths)
    return akurasi_df

In [ ]:
plt.figure(figsize=(15,3))
sns.lineplot(data=akurasi_df, marker='o', palette='viridis')
plt.legend()
plt.grid()
plt.title('Akurasi Train dan Test berdasarkan depth')
plt.xlabel('Depths')
plt.ylabel('Akurasi')
plt.show()